# E09: Suelo, ráster y profundidad

[Enunciado](../../talleres/E09.md) · [Entorno WSL](../../docs/instalacion-wsl.md)

Ejecutar en orden con el kernel del curso. Preparar las fuentes indicadas en el enunciado; no se descargan automáticamente. Las salidas se regeneran en `kit/salidas/`. Los originales se conservan.
[Linux/macOS](../../docs/instalacion-linux-macos.md). Ajustar la ruta explícita. Las operaciones se muestran directamente con pandas, GeoPandas y Rasterio; las funciones del kit quedan como referencia posterior de automatización.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

# Editar esta ruta para que coincida con el clon local.
# Linux/macOS: Path.home() / "bigdata/big-data-postgraduate"
repositorio = Path("/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate")
KIT = repositorio / "kit"
RAW = KIT / "data/raw"
OUT = KIT / "salidas" / "E09"
assert KIT.is_dir(), "Corrige repositorio antes de continuar"
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show
from shapely.geometry import box, shape
from IPython.display import Image


## 1. Leer el ráster, revisar metadatos y decidir la máscara
Preparar SoilGrids, WoSIS y los dos archivos IGAC según E09. `read(masked=True)` respeta NoData. Contar ceros antes de excluirlos provisionalmente del promedio; no declararlos NoData en el original. Aplicar el factor valor/10 para obtener pH de 0–5 cm.


In [ ]:
with rasterio.open(RAW / 'soilgrids.tif') as src:
    valores = src.read(1, masked=True).astype(float)
    transformacion = src.transform
    crs_suelo = src.crs
    limites_suelo = src.bounds
    nodata = src.nodata
assert crs_suelo is not None
ceros = int(np.sum(valores.filled(np.nan) == 0))
ph = np.ma.masked_where((valores <= 0) | (valores > 140), valores) / 10
control = {'soilgrids': {'shape': list(valores.shape), 'crs': str(crs_suelo), 'nodata': nodata,
    'celdas_cero': ceros, 'pixeles_validos': int(ph.count()), 'min_ph': float(ph.min()),
    'max_ph': float(ph.max()), 'media_ph_sin_ceros_por_revisar': float(ph.mean())}}
assert valores.shape == (42, 44) and ceros == 188 and ph.count() == 1660
display(control)


### Leer WoSIS y dibujar el recorte
Transformar los perfiles al CRS del ráster y verificar si intersectan su extensión antes de dibujarlos.


In [ ]:
perfiles = gpd.read_file(RAW / 'wosis.json', engine='pyogrio')
assert perfiles.crs is not None
perfiles = perfiles.to_crs(crs_suelo)
dentro = perfiles.geometry.intersects(box(*limites_suelo))
fig, ax = plt.subplots(figsize=(8, 6))
show(ph, transform=transformacion, ax=ax, cmap='viridis')
fig.colorbar(ax.images[0], ax=ax, label='pH (valor / 10)')
if dentro.any():
    perfiles.loc[dentro].plot(ax=ax, color='red', markersize=15)
ax.set_title(f'SoilGrids 0–5 cm: {int(dentro.sum())} horizontes WoSIS dentro del recorte')
fig.savefig(OUT / 'mapa_suelo.png', dpi=160, bbox_inches='tight')
plt.close(fig)
perfiles.drop(columns='geometry').to_csv(OUT / 'perfiles_wosis.csv', index=False)
control['wosis'] = {'registros': len(perfiles), 'perfiles_distintos': int(perfiles.profile_id.nunique())}
with open(OUT / 'control_suelo.json', 'w', encoding='utf-8') as archivo:
    json.dump(control, archivo, ensure_ascii=False, indent=2)
display(Image(filename=str(OUT / 'mapa_suelo.png')))


## 2. Perfiles y horizontes
Diez registros no significan diez sitios: distinguir perfil de horizonte. Revisar método, profundidad e incertidumbre de ubicación antes de comparar con una predicción.

In [ ]:
wosis=gpd.read_file(RAW / 'wosis.json', engine='pyogrio')
assert len(wosis)==10 and wosis.profile_id.nunique()==3
display(wosis[['profile_id','upper_depth','lower_depth','value_avg','method_options','positional_uncertainty']].sort_values(['profile_id','upper_depth']))
import rasterio
from shapely.geometry import box
with rasterio.open(RAW/'soilgrids.tif') as src:
    dentro=wosis.to_crs(src.crs).intersects(box(*src.bounds))
print('Horizontes dentro del recorte:',int(dentro.sum()))

## 3. Capas temáticas y límites de interpretación
Mostrar los atributos de química y correlación para distinguir intervalos cartográficos y taxonomía. No calcular error de predicción sin coincidencia espacial, profundidad y método compatibles.

In [ ]:
capa=gpd.read_file(RAW / 'igac_quimica.json', engine='pyogrio')
print('IGAC química: CRS',capa.crs,'extensión',capa.total_bounds)
display(capa.drop(columns='geometry'))
# Correlación es una tabla ArcGIS de atributos, no una capa con geometría.
correlacion=json.loads((RAW/'igac_correlacion.json').read_text())
tabla_correlacion=pd.DataFrame([f['attributes'] for f in correlacion['features']])
assert len(tabla_correlacion)==10
display(tabla_correlacion)


**Entrega:** mapa, control_suelo.json y tabla por perfil/profundidad. Los 188 ceros no tienen NoData declarado: se separan provisionalmente para la media, conservando el original. El factor de escala es valor/10. Documentar las condiciones que faltan para validar la predicción; no interpretar el recorte como análisis de finca.